In [3]:
import pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "parquet"
from PyDI.io import load_parquet

amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_columns', None)



In [47]:
temporary = amazon_sample.merge(
    goodreads_sample,
    on="isbn_clean",
    suffixes=("_a", "_g"),
)
merged = temporary.merge(
    metabooks_sample,
    on="isbn_clean",
    suffixes=("", "_m"),  # give metabooks columns the _mbk tag
)

# group common fields side by side: [base_amz, base_gr, base_mbk]
bases = ["id","title", "author", "clean_author", "clean_title", "clean_publisher",
         "language", "genres", "publisher", "publish_year", "page_count",
         "price", "rating", "numratings", "bookformat", "edition"]
# always keep the join key first
ordered = ["isbn_clean"]

for base in bases:
    cols = [c for c in merged.columns if c.startswith(base)]
    ordered.extend(cols)

# add any leftover columns not covered above
ordered.extend([c for c in merged.columns if c not in ordered])

merged = merged[ordered]

merged.rename(
    columns={
        "title": "title_m",
        "author": "author_m",
        "language": "language_g",
        "publisher": "publisher_m",
        "genres": "genres_g",
        "publish_year": "publish_year_m",
        "page_count": "page_count_g",
        "price": "price_g",
        "rating": "rating_g",
        "numratings": "numratings_g",
        "bookformat": "bookformat_g",
        "edition": "edition_g",
        "id": "id_m",
    },
    inplace=True,
)

In [48]:
merged.drop(columns=["bookformat_g", "edition_g","rating_g","rating_m","numratings_g","numratings_m"], inplace=True)

In [51]:
# I need to set a seed for reproducibility
merged.sample(20,random_state=42).to_csv((ROOT/"golden_merged.csv"), index=False)